# ArtSense AI — Training Notebook
Fine-tune EfficientNet-B3 on the [Best Artworks of All Time](https://www.kaggle.com/datasets/ikarus777/best-artworks-of-all-time) dataset (50 artists, ~8,000 images).

**Output files:**
- `efficientnet_b3_artsense.pth` — model weights
- `class_indices.json` — `{artist_name: class_index}` mapping

After training, download both files and place them in the `model/` folder of your local project.

## 1. Install & Import

In [ ]:
!pip install -q rapidfuzz

import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, models, transforms
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## 2. Dataset Paths
The Kaggle dataset `ikarus777/best-artworks-of-all-time` is mounted at `/kaggle/input/best-artworks-of-all-time/`.  
Make sure you have added it via **Add Data** before running.

In [ ]:
DATA_ROOT = Path("/kaggle/input/best-artworks-of-all-time")
IMAGES_DIR = DATA_ROOT / "images" / "images"   # images are nested one extra level on Kaggle
CSV_PATH   = DATA_ROOT / "artists.csv"

# Verify paths
print("Images dir exists:", IMAGES_DIR.exists())
print("CSV exists:", CSV_PATH.exists())

artist_folders = sorted([d for d in IMAGES_DIR.iterdir() if d.is_dir()])
print(f"\nNumber of artist folders found: {len(artist_folders)}")
for folder in artist_folders:
    count = len(list(folder.glob("*.jpg")))
    print(f"  {folder.name}: {count} images")

## 3. Build Image List & Class Index

In [ ]:
# Build a flat list of (image_path, class_index) and save class mapping
all_images = []   # list of (path_str, label_int)
class_to_idx = {}

for idx, folder in enumerate(artist_folders):
    class_name = folder.name
    class_to_idx[class_name] = idx
    for img_path in folder.glob("*.jpg"):
        all_images.append((str(img_path), idx))

# Save class index mapping — we need this in the Gradio app
with open("class_indices.json", "w") as f:
    json.dump(class_to_idx, f, indent=2)

print(f"Total images: {len(all_images)}")
print(f"Total classes: {len(class_to_idx)}")
print("\nSample entries:", all_images[:3])

## 4. Train/Val Split & Dataset Class

In [ ]:
from torch.utils.data import Dataset

paths  = [x[0] for x in all_images]
labels = [x[1] for x in all_images]

train_paths, val_paths, train_labels, val_labels = train_test_split(
    paths, labels, test_size=0.2, stratify=labels, random_state=SEED
)
print(f"Train: {len(train_paths)} | Val: {len(val_paths)}")

# Transforms
train_transform = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class ArtDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        image = Image.open(self.paths[idx]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, self.labels[idx]

train_dataset = ArtDataset(train_paths, train_labels, train_transform)
val_dataset   = ArtDataset(val_paths,   val_labels,   val_transform)

## 5. Weighted Sampler (fix class imbalance)

In [ ]:
from collections import Counter

label_counts = Counter(train_labels)
class_weights = {cls: 1.0 / count for cls, count in label_counts.items()}
sample_weights = [class_weights[lbl] for lbl in train_labels]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True,
)

train_loader = DataLoader(train_dataset, batch_size=32, sampler=sampler,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False,    num_workers=2, pin_memory=True)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

## 6. Build Model (EfficientNet-B3)

In [ ]:
NUM_CLASSES = len(class_to_idx)

model = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.DEFAULT)

# Freeze backbone
for param in model.parameters():
    param.requires_grad = False

# Replace classifier head
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, NUM_CLASSES)

model = model.to(DEVICE)
print(f"Model ready — {NUM_CLASSES} output classes")
print(f"Classifier head: {model.classifier[1]}")

## 7. Training Helper

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    """Run one epoch. If optimizer is None, run in eval mode (no gradients)."""
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if is_train else torch.no_grad()

    with ctx:
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += images.size(0)

    return total_loss / total, correct / total

## 8. Phase 1 Training — Head Only (backbone frozen, 5 epochs)

In [ ]:
criterion = nn.CrossEntropyLoss()

# Only train the new head
optimizer = optim.AdamW(model.classifier.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5)

PHASE1_EPOCHS = 5
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

print("Phase 1: Training classifier head (backbone frozen)")
for epoch in range(1, PHASE1_EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer)
    vl_loss, vl_acc = run_epoch(model, val_loader,   criterion)
    scheduler.step()

    history["train_loss"].append(tr_loss)
    history["train_acc"].append(tr_acc)
    history["val_loss"].append(vl_loss)
    history["val_acc"].append(vl_acc)

    print(f"Epoch {epoch}/{PHASE1_EPOCHS} | "
          f"Train Loss: {tr_loss:.4f} Acc: {tr_acc:.3f} | "
          f"Val Loss: {vl_loss:.4f} Acc: {vl_acc:.3f}")

## 9. Phase 2 Training — Full Fine-tune (all layers, 15 epochs)

In [ ]:
# Unfreeze all layers
for param in model.parameters():
    param.requires_grad = True

PHASE2_EPOCHS = 15
optimizer2 = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler2 = optim.lr_scheduler.CosineAnnealingLR(optimizer2, T_max=PHASE2_EPOCHS)

best_val_acc = 0.0

print("Phase 2: Fine-tuning all layers")
for epoch in range(1, PHASE2_EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer2)
    vl_loss, vl_acc = run_epoch(model, val_loader,   criterion)
    scheduler2.step()

    history["train_loss"].append(tr_loss)
    history["train_acc"].append(tr_acc)
    history["val_loss"].append(vl_loss)
    history["val_acc"].append(vl_acc)

    print(f"Epoch {epoch}/{PHASE2_EPOCHS} | "
          f"Train Loss: {tr_loss:.4f} Acc: {tr_acc:.3f} | "
          f"Val Loss: {vl_loss:.4f} Acc: {vl_acc:.3f}", end="")

    # Save best model checkpoint
    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(model.state_dict(), "efficientnet_b3_artsense.pth")
        print("  <-- saved best model", end="")
    print()

print(f"\nBest validation accuracy: {best_val_acc:.3f}")

## 10. Plot Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, len(history["train_loss"]) + 1)

ax1.plot(epochs, history["train_loss"], label="Train")
ax1.plot(epochs, history["val_loss"],   label="Val")
ax1.axvline(x=PHASE1_EPOCHS + 0.5, color="gray", linestyle="--", label="Unfreeze")
ax1.set_title("Loss")
ax1.set_xlabel("Epoch")
ax1.legend()

ax2.plot(epochs, history["train_acc"], label="Train")
ax2.plot(epochs, history["val_acc"],   label="Val")
ax2.axvline(x=PHASE1_EPOCHS + 0.5, color="gray", linestyle="--", label="Unfreeze")
ax2.set_title("Accuracy")
ax2.set_xlabel("Epoch")
ax2.legend()

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()

## 11. Download Output Files

Run the cell below, then use Kaggle's output panel on the right to download both files.

Place them in your local project:
- `efficientnet_b3_artsense.pth` → `model/`
- `class_indices.json` → `model/`

In [ ]:
import os
print("Output files:")
for fname in ["efficientnet_b3_artsense.pth", "class_indices.json", "training_curves.png"]:
    size = os.path.getsize(fname) / 1e6 if os.path.exists(fname) else 0
    status = f"{size:.1f} MB" if size > 0 else "NOT FOUND"
    print(f"  {fname}: {status}")